## Variable jit settings

It seems that only the jit applied to the final plum.dispatch function matters for whether a set of functions are compiled or not. 
And critically, the settings for the jit are shared across all functions in the dispatch.
This is a major limitation when I want to use different jit settings for different functions in the dispatch, or want to compile some functions but not others.

And I can't dispatch over compiled functions, because plum.dispatch rejects compiled functions.

Solution isn't too bad: just define functions with separate names and unique compilation settings. 
Call them with same-named function.

Downside: I can't as easily turn compilation for the given function, maybe making debugging harder. Jax provides a disable_jit function. Okay, I'll just test using that whenever I need to, and go ahead and add jit to my library where appropriate.

In [1]:
import jax
import jax.numpy as jnp
from jaxcmr.models import LinearAssociativeMemory

item_count = 10
learning_rate = .1

with jax.disable_jit():
    @jax.jit
    def basic_init_mfc(
        item_count,
        learning_rate
        ) -> LinearAssociativeMemory.LinearAssociativeMfc:
        "A linear associative feature-to-context memory assuming one-hot item representations"
        memory = jnp.eye(item_count, item_count + 2)
        memory = jnp.hstack([jnp.zeros((item_count, 1)), memory[:, :-1]])
        memory = memory * (1 - learning_rate)
        return LinearAssociativeMemory.LinearAssociativeMfc(memory)

    basic_init_mfc(item_count, learning_rate)

basic_init_mfc(item_count, learning_rate)

TypeError: Shapes must be 1D sequences of concrete values of integer type, got (Traced<ShapedArray(int32[], weak_type=True)>with<DynamicJaxprTrace(level=1/0)>,). 'N' argument of jnp.eye().
If using `jit`, try using `static_argnums` or applying `jit` to smaller subfunctions.
The error occurred while tracing the function basic_init_mfc at /tmp/ipykernel_38981/2133557467.py:9 for jit. This concrete value was not available in Python because it depends on the value of the argument item_count.